# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata.to_json()

print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Dataset Identifier: {dataset.metadata.identifier}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Each record set, field, and column is referenced by its unique `@id`. Let's enumerate what is available in this dataset.

In [ ]:
# List record sets and their fields using `@id`
record_sets_info = []
for rs in dataset.metadata.record_set:
    fields = rs.field
    print(f"RecordSet @id: {rs['@id']}, Name: {getattr(rs, 'name', 'N/A')}")
    print("  Fields:")
    for field in fields:
        print(f"    Field @id: {field['@id']}, Name: {getattr(field, 'name', 'N/A')}, DataType: {getattr(field, 'data_type', 'N/A')}")
    record_sets_info.append({
        'record_set_id': rs['@id'],
        'fields': [field['@id'] for field in fields],
        'field_names': [getattr(field, 'name', 'N/A') for field in fields]
    })

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

We will extract data from all main record sets. Each record set and field will be referenced by its `@id`.

In [ ]:
# Extract all record set IDs
record_sets = [info['record_set_id'] for info in record_sets_info]
dataframes = {}

for record_set in record_sets:
    records = list(dataset.records(record_set=record_set))
    # Load these as a pandas DataFrame
    dataframes[record_set] = pd.DataFrame(records)

# Show the columns for the first record set
first_rs = record_sets[0] if len(record_sets) > 0 else None
if first_rs:
    print(f"Available columns for record set {first_rs}:")
    print(dataframes[first_rs].columns.tolist())
    display(dataframes[first_rs].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Below we select a numeric field (such as GAD-7 score) by its field `@id` for analysis.

In [ ]:
# Example: Select a numeric field (e.g., GAD-7 score)
# Find a numeric field's @id.
selected_record_set = first_rs  # the first record set for demonstration
numeric_field_id = None
group_field_id = None

# Find a field with 'GAD' or similar in its name and 'Integer' or 'Float' data type
for info in record_sets_info:
    if info['record_set_id'] == selected_record_set:
        field_objs = getattr(dataset.metadata, 'record_set')[record_sets.index(selected_record_set)].field
        for field in field_objs:
            field_name = getattr(field, 'name', '')
            data_type = getattr(field, 'data_type', '')
            if ('GAD' in field_name or 'phq' in field_name.lower() or 'psq' in field_name.lower()) and data_type in ['Integer', 'Float', 'Number']:
                numeric_field_id = field['@id']
            if 'education' in field_name.lower() or 'gender' in field_name.lower():
                group_field_id = field['@id']

# Ensure we found fields
if numeric_field_id is not None:
    print(f"Using numeric field @id: {numeric_field_id}")
    df = dataframes[selected_record_set]

    if numeric_field_id in df.columns:
        # Filter records with score > threshold (example threshold)
        threshold = 10  # e.g., moderate anxiety
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalization
        col_norm = f"{numeric_field_id}_normalized"
        filtered_df[col_norm] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, col_norm]].head())

        # Grouping
        if group_field_id is not None and group_field_id in df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"Grouped mean {numeric_field_id} by {group_field_id}:")
            display(grouped_df.head())
else:
    print("No suitable numeric field found for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we show a histogram of the selected numeric score field, and a box plot by the grouping variable if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of {numeric_field_id} scores')
    plt.xlabel('Score')
    plt.ylabel('Frequency')
    plt.show()

    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(8, 4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f'{numeric_field_id} Scores by {group_field_id}')
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset provides detailed survey responses from Kilifi County, including standardized mental health scores and demographics, with all entities referenced by their `@id`.
- Data was effectively loaded, overviewed, and visualized using `mlcroissant`, supporting reproducible, AI-ready workflows.
- Filtering and normalization steps enable deeper insights into mental health distributions. More advanced analytics can be performed using the rich metadata schema.
- For more advanced integration or data wrangling, use the `mlcroissant` API with `@id` references as shown throughout.